# Ozone + OIDC — a guided tour

This notebook walks through every authentication flow of
[`ozone-oidc-proxy`](https://github.com/aimd54/ozone-oidc-proxy): a reverse
proxy that puts **OIDC authentication** in front of a stock, unsecured
**Apache Ozone S3 Gateway** — JWTs in, temporary AWS-style credentials out,
Ozone's own ACLs deciding who may touch what.

It is designed to run **inside the `jupyter` container** of the compose
stack (`make up && make init && make lakehouse-up`), where the service
hostnames below resolve. Sections 9 and 10 need the optional overlays
(`make edge-up`, `make lakehouse-up`) and skip gracefully without them.

| Lane | What happens |
| --- | --- |
| **STS** | `POST /` with `Action=AssumeRoleWithWebIdentity`: JWT → temp credentials (`OZPX...`) |
| **SigV4** | normal AWS clients sign with those credentials; the proxy re-verifies every request |
| **Presigned** | query-auth URLs for credential-less consumers |
| **Bearer** | a JWT directly on the request (secondary; disable in production) |

Authoritative docs: `docs/DESIGN.md` (architecture), `docs/PRODUCTION.md`
(hardening checklist), `docs/UPSTREAM.md` (the native-Ozone future).

In [ ]:
import base64
import json
import uuid
from xml.etree import ElementTree as ET

import boto3
import requests
from botocore.config import Config
from botocore.exceptions import ClientError

PROXY = "http://proxy:9000"        # S3 + STS — the only sanctioned door to Ozone
ADMIN = "http://proxy:9090"        # health / metrics / revocation (operators only)
KEYCLOAK = "http://keycloak:8080/realms/ozone"
EDGE = "https://haproxy:8443"      # TLS edge overlay (make edge-up)
NESSIE = "http://nessie:19120"     # lakehouse overlay (make lakehouse-up)
ROLE_ARN = "arn:ozone:iam::dev:role/oidc"
REGION = "us-east-1"

run_id = uuid.uuid4().hex[:8]      # keeps re-runs from colliding
print("tour run id:", run_id)

## 1 — Authenticate: get an OIDC JWT

The proxy never sees a password: identity comes from the IdP (Keycloak
here). This notebook uses the **ROPC grant for lab convenience only** —
humans use the `ozone-login` device flow or the credential portal, and
workloads use the client-credentials grant (section 10 shows one live).

In [ ]:
def get_token(username, password):
    r = requests.post(
        f"{KEYCLOAK}/protocol/openid-connect/token",
        data={"grant_type": "password", "client_id": "ozone-s3",
              "username": username, "password": password},
        timeout=10)
    r.raise_for_status()
    return r.json()["access_token"]

alice_jwt = get_token("alice", "password123")
claims = json.loads(base64.urlsafe_b64decode(alice_jwt.split(".")[1] + "=="))
{k: claims[k] for k in ("iss", "aud", "preferred_username", "exp")}

## 2 — Exchange it: `AssumeRoleWithWebIdentity`

The STS lane validates the JWT (issuer allowlist, signature via JWKS,
`aud`, expiry) and mints **temporary AWS-style credentials** held in the
proxy's store. TTL = min(token expiry, requested duration, server max).

In [ ]:
def sts_exchange(jwt, endpoint=PROXY):
    r = requests.post(endpoint + "/", data={
        "Action": "AssumeRoleWithWebIdentity", "Version": "2011-06-15",
        "RoleArn": ROLE_ARN, "RoleSessionName": "notebook",
        "WebIdentityToken": jwt}, timeout=10)
    r.raise_for_status()
    creds = ET.fromstring(r.text).find(".//{*}Credentials")
    return {el.tag.split("}")[1]: el.text for el in creds}

alice = sts_exchange(alice_jwt)
print("AccessKeyId:", alice["AccessKeyId"], "— expires", alice["Expiration"])

## 3 — Use them: the SigV4 data path

Any AWS client works unchanged. The proxy re-derives every request
signature from the wire and forwards to Ozone with the request attributed
to the OIDC username — `ozone sh bucket info /s3v/<bucket>` on the OM shows
**owner = alice**, and Ozone's native ACLs apply to her.

In [ ]:
def s3_client(creds, endpoint=PROXY, **kw):
    # signature_version s3v4: boto3 presigns with the legacy v2 query
    # scheme by default, which the proxy's strict mode rightly rejects.
    return boto3.client(
        "s3", endpoint_url=endpoint, region_name=REGION,
        config=Config(signature_version="s3v4",
                      s3={"addressing_style": "path"}),
        aws_access_key_id=creds["AccessKeyId"],
        aws_secret_access_key=creds["SecretAccessKey"],
        aws_session_token=creds["SessionToken"], **kw)

s3 = s3_client(alice)
bucket = f"tour-{run_id}"
s3.create_bucket(Bucket=bucket)
s3.put_object(Bucket=bucket, Key="hello.txt", Body=b"hello from the oidc tour")
print(s3.get_object(Bucket=bucket, Key="hello.txt")["Body"].read().decode())
print([o["Key"] for o in s3.list_objects_v2(Bucket=bucket)["Contents"]])

## 4 — Zero-config clients: the SDK auto-exchange (workload identity)

AWS SDKs can do the whole exchange **by themselves** from three environment
variables — no access keys anywhere in the client. This is exactly how the
Nessie server authenticates in section 10, and what `ozone-login` maintains
for humans (it keeps the token file fresh).

In [ ]:
import os
import pathlib

pathlib.Path("/tmp/alice.jwt").write_text(alice_jwt)
os.environ.update({
    "AWS_ROLE_ARN": ROLE_ARN,
    "AWS_WEB_IDENTITY_TOKEN_FILE": "/tmp/alice.jwt",
    "AWS_ROLE_SESSION_NAME": "notebook-auto",
    "AWS_ENDPOINT_URL_STS": PROXY,
    "AWS_DEFAULT_REGION": REGION,
})
auto = boto3.Session()
print("SDK auto-minted AKID:",
      auto.get_credentials().get_frozen_credentials().access_key)
print("objects seen with zero explicit credentials:",
      auto.client("s3", endpoint_url=PROXY)
          .list_objects_v2(Bucket=bucket)["KeyCount"])

## 5 — Presigned URLs: sharing without credentials

A presigned URL carries the SigV4 signature in the query string; the
fetcher needs nothing. The proxy verifies it on the wire form — tampering
with as little as one query parameter breaks the signature.

In [ ]:
url = s3.generate_presigned_url(
    "get_object", Params={"Bucket": bucket, "Key": "hello.txt"}, ExpiresIn=120)
print("anonymous fetch:", requests.get(url, timeout=10).text)

tampered = requests.get(url + "&admin=true", timeout=10)
print("tampered fetch:", tampered.status_code,
      "SignatureDoesNotMatch" in tampered.text and "(SignatureDoesNotMatch)")

## 6 — The Bearer lane (secondary)

A JWT straight on the request. Convenient for quick scripts and the
oauth2-proxy browser lane — but it puts a long-lived token on every call,
so `docs/PRODUCTION.md` recommends disabling it in production
(`data_path.accept_bearer: false`).

In [ ]:
r = requests.get(f"{PROXY}/{bucket}/hello.txt",
                 headers={"Authorization": f"Bearer {alice_jwt}"}, timeout=10)
print(r.status_code, r.text)

## 7 — Authorization stays Ozone's: the ACL matrix

Authentication (the proxy) and authorization (Ozone) are separate. bob
authenticates fine — and is still denied on alice's bucket until someone
grants him access:

```bash
docker compose exec ozone-om ozone sh bucket addacl -a user:bob:rl /s3v/<bucket>
```

In [ ]:
bob = sts_exchange(get_token("bob", "password123"))
try:
    s3_client(bob).list_objects_v2(Bucket=bucket)
    print("UNEXPECTED: bob was allowed")
except ClientError as e:
    print("bob denied as designed:", e.response["Error"]["Code"])

## 8 — Revocation: kill credentials before they expire

The admin surface (never exposed beyond operators) can delete any minted
credential; with the valkey store this propagates to every proxy replica
within milliseconds.

In [ ]:
victim = sts_exchange(alice_jwt)
sv = s3_client(victim)
sv.list_objects_v2(Bucket=bucket)          # works...
code = requests.delete(f"{ADMIN}/credentials/{victim['AccessKeyId']}",
                       timeout=10).status_code
print("revocation HTTP status:", code)
try:
    sv.list_objects_v2(Bucket=bucket)      # ...and now it does not
    print("UNEXPECTED: revoked credentials still work")
except ClientError as e:
    print("revoked as designed:", e.response["Error"]["Code"])

## 9 — The TLS edge (optional: `make edge-up`)

Production runs behind an HAProxy edge terminating TLS; the overlay models
it with a self-signed certificate. SigV4 signs the `Host` header, so the
edge forwards it untouched and signatures minted for `haproxy:8443` verify.
(`verify=False` here because the lab cert is self-signed — real
deployments pin the real CA.)

In [ ]:
try:
    requests.get(EDGE, verify=False, timeout=3)
    edge_up = True
except requests.RequestException:
    edge_up = False

if edge_up:
    import urllib3
    urllib3.disable_warnings()
    s3_edge = s3_client(alice, endpoint=EDGE, verify=False)
    print("objects via https through HAProxy:",
          s3_edge.list_objects_v2(Bucket=bucket)["KeyCount"])
else:
    print("edge overlay not running — make edge-up (section skipped)")

## 10 — Iceberg + Nessie: a lakehouse on OIDC credentials (optional: `make lakehouse-up`)

Two identities cooperate on one Iceberg table, both via OIDC:

- **Nessie** (the catalog server) holds **no static S3 secret at all** —
  a sidecar keeps a Keycloak *client-credentials* JWT fresh, and the AWS
  SDK inside Nessie auto-exchanges it at the proxy STS
  (`service-account-nessie`), exactly like section 4.
- **alice** writes the table's data files with her own temp credentials
  via PyIceberg against Nessie's Iceberg REST endpoint.

In [ ]:
try:
    requests.get(f"{NESSIE}/api/v2/config", timeout=3)
    nessie_up = True
except requests.RequestException:
    nessie_up = False
print("nessie:", "up" if nessie_up else
      "not running — make lakehouse-up (sections below skip)")

In [ ]:
if nessie_up:
    import pyarrow as pa
    from pyiceberg.catalog import load_catalog

    catalog = load_catalog("nessie", **{
        "type": "rest",
        "uri": f"{NESSIE}/iceberg/main/",
        "s3.endpoint": PROXY,
        "s3.access-key-id": alice["AccessKeyId"],
        "s3.secret-access-key": alice["SecretAccessKey"],
        "s3.session-token": alice["SessionToken"],
        "s3.region": REGION,
        "s3.path-style-access": "true",
    })
    ns = f"tour_{run_id}"
    catalog.create_namespace(ns)
    table = catalog.create_table(f"{ns}.events", schema=pa.schema([
        ("id", pa.int64()), ("who", pa.string()), ("what", pa.string())]))
    table.append(pa.table({"id": [1, 2, 3], "who": ["alice"] * 3,
                           "what": ["login", "exchange", "query"]}))
    print(table.scan().to_pandas())

Every Iceberg commit is a Nessie commit — git-like history over the
catalog, while the bytes live in `s3://lakehouse/` on Ozone, written under
OIDC-minted credentials:

In [ ]:
if nessie_up:
    history = requests.get(f"{NESSIE}/api/v2/trees/main/history",
                           timeout=10).json()
    for commit in history.get("logEntries", [])[:5]:
        meta = commit["commitMeta"]
        print(meta["commitTime"], "—", meta["message"])

## Wrap-up

You just used one identity system for: human S3 access, zero-config SDK
workloads, anonymous sharing, browser-style Bearer calls, live revocation,
a TLS production edge, and an Iceberg lakehouse whose catalog server owns
no secret at all.

Where to go next:

- `docs/DESIGN.md` — the architecture these lanes implement
- `docs/PRODUCTION.md` — what separates this lab from production
- `docs/VERIFICATION.md` — the live acceptance record
- `docs/UPSTREAM.md` — the native-Ozone future and the migration path